<a href="https://colab.research.google.com/github/BandiSreesaicharan/NLP/blob/main/NLP_Lab_Assignment_11_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
df = pd.read_csv('/content/movie_review.csv')
df = df.drop(columns=['sent_id','html_id','cv_tag','fold_id'])
df.head()

,text,tag
0,films adapted from comic books have had plenty...,pos
1,"for starters , it was created by alan moore ( ...",pos
2,to say moore and campbell thoroughly researche...,pos
3,"the book ( or "" graphic novel , "" if you will ...",pos
4,"in other words , don't dismiss this film becau...",pos


In [11]:
import re
import nltk

from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer

# Download required NLTK resources (run once)
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [16]:
# Text Preprocessing Pipeline
def nltk_preprocessing_pipeline(text):

    lemmatizer = WordNetLemmatizer()
    stop_words = set(stopwords.words('english'))

    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)

    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)

    # Remove mentions
    text = re.sub(r'@\w+', '', text)

    # Remove hashtags
    text = re.sub(r'#\w+', '', text)

    # Convert to lowercase
    text = text.lower()

    # Remove emojis
    emoji_pattern = re.compile(
        "["
        "\U0001F600-\U0001F64F"
        "\U0001F300-\U0001F5FF"
        "\U0001F680-\U0001F6FF"
        "\U0001F1E0-\U0001F1FF"
        "\U00002702-\U000027B0"
        "\U000024C2-\U0001F251"
        "]+",
        flags=re.UNICODE
    )

    text = emoji_pattern.sub(r'', text)

    # Remove special characters
    text = re.sub(r'[^a-zA-Z0-9\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # Word Tokenization
    tokenized_words = word_tokenize(text)

    # Stopword Removal
    stop_words = set(stopwords.words('english'))
    filtered_words = [word for word in tokenized_words if word not in stop_words]

    # Lemmatization
    lemmatizer = WordNetLemmatizer()
    lemmatized_words = [lemmatizer.lemmatize(word) for word in filtered_words]

    # Join words
    clean_text = " ".join(lemmatized_words)

    return clean_text

df["text"] = df["text"].apply(nltk_preprocessing_pipeline)
df.head()

,text,tag
0,film adapted comic book plenty success whether...,pos
1,starter created alan moore eddie campbell brou...,pos
2,say moore campbell thoroughly researched subje...,pos
3,book graphic novel 500 page long includes near...,pos
4,word dont dismiss film source,pos


In [19]:
# TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000)
X_tfidf = tfidf.fit_transform(df["text"])

# Display Vocabulary Size
vocab_size = len(tfidf.vocabulary_)

print("\nVocabulary Size:", vocab_size)

# Show some vocabulary words
print("\nSample vocabulary words:")
print(list(tfidf.vocabulary_.keys())[:20])

# TF-IDF Matrix Shape
print("\nTF-IDF Matrix Shape:", X_tfidf.shape)


Vocabulary Size: 5000

Sample vocabulary words:
['film', 'adapted', 'comic', 'book', 'plenty', 'success', 'whether', 'theyre', 'batman', 'superman', 'spawn', 'toward', 'kid', 'casper', 'crowd', 'ghost', 'world', 'there', 'never', 'really']

TF-IDF Matrix Shape: (64720, 5000)


In [20]:
#Train Test Split
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X_tfidf, y, test_size=0.2, random_state=42
)

In [21]:
#Traning
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

In [23]:
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from sklearn.metrics import confusion_matrix
#Accuracy
accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

#Classification Report
print(classification_report(y_test, y_pred))

#Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
print(cm)

Accuracy: 0.670040173053152
              precision    recall  f1-score   support

           0       0.67      0.66      0.66      6371
           1       0.67      0.68      0.68      6573

    accuracy                           0.67     12944
   macro avg       0.67      0.67      0.67     12944
weighted avg       0.67      0.67      0.67     12944

[[4186 2185]
 [2086 4487]]
